# Cholesky Decomposition

## Introduction

Let $\mathbf{A}$ be a square matrix. Then, $\mathbf{A}$ can be represented as

$$ \mathbf{A} = \mathbf{LU} $$

where $\mathbf{L}$ and $\mathbf{U}$ are lower and upper triangular matrices, respectively. Such a representation is not unique. 

Cholesky decomposition is a method to represent $\mathbf{A}$ as $\mathbf{LL^T}$, where $\mathbf{L}$ is a lower triangular matrix (and so $\mathbf{L^T}$ is an upper triangular matrix). However, $\mathbf{LL^T}$ is a symmetric matrix, and thus, $\mathbf{A}$ must be symmetric for such a decomposition to be possible. Further, $\mathbf{A}$ must be positive definite for Cholesky decomposition. 

## Method

Let $\mathbf{A}$ be a $3 \times 3$ matrix:

$$
\mathbf{A} = 
\begin{pmatrix}
A_{11} & A_{12} && A_{13} \\
A_{21} & A_{22} && A_{23} \\
A_{31} & A_{32} && A_{33} \\
\end{pmatrix}
$$

Then, for the Cholesky decomposition, we have:

$$
\begin{pmatrix}
A_{11} & A_{12} && A_{13} \\
A_{21} & A_{22} && A_{23} \\
A_{31} & A_{32} && A_{33} \\
\end{pmatrix}
=
\begin{pmatrix}
L_{11} & 0 && 0 \\
L_{21} & L_{22} && 0 \\
L_{31} & L_{32} && L_{33} \\
\end{pmatrix}
\begin{pmatrix}
L_{11} & L_{21} && L_{31} \\
0 & L_{22} && L_{32} \\
0 & 0 && L_{33} \\
\end{pmatrix}
$$

That is:

$$
\begin{pmatrix}
A_{11} & A_{12} && A_{13} \\
A_{21} & A_{22} && A_{23} \\
A_{31} & A_{32} && A_{33} \\
\end{pmatrix}
=
\begin{pmatrix}
L_{11}^2 & L_{11}L_{21} && L_{11}L_{31} \\
L_{11}L_{21} & L_{21}^2 + L_{22}^2 && L_{21}L_{31} + L_{22}L_{32} \\
L_{11}L_{31} & L_{21}L_{31} + L_{22}L_{32} && L_{31}^2 + L_{32}^2 + L_{33}^2\\
\end{pmatrix}
$$

We thus obtain six equations (not nine, due to symmetry), and can solve them as follows:

$$A_{11} = L_{11}^2 \implies L_{11} = \sqrt{A_{11}}$$
$$A_{21} = L_{11}L_{21} \implies L_{21} = A_{21} / L_{11}$$
$$A_{31} = L_{11}L_{31} \implies L_{31} = A_{31} / L_{11}$$
$$A_{22} = L_{21}^2 + L_{22}^2 \implies L_{22} = \sqrt{A_{22} - L_{21}^2}$$
$$A_{32} = L_{21}L_{31} + L_{22}L_{32} \implies L_{32} = (A_{32} - L_{21}L_{31})/L_{22}$$
$$A_{33} = L_{31}^2 + L_{32}^2 + L_{33}^2 \implies L_{33} = \sqrt{A_{33} - L^2_{31} - L^{2}_{32}}$$

(Note that we consider the positive square root here.)

Similarly, for an $n \times n$ matrix, we have:

$$ L_{j, j} = \sqrt{A_{j, j} - \sum_{k = 1}^{j - 1} L^2_{j, k}} $$

and, for $i > j$, 

$$ L_{i, j} = \frac{1}{L_{j, j}} \left(A_{i, j} - \sum_{k = 1}^{j - 1} L_{i, k}L_{j, k}\right). $$

## Codes

### 1. Python

We first consider the following code, written largely in Python. That is, it avoids the speedup one can obtain using vectorized operations in NumPy. 

In [1]:
import numpy as np

In [2]:
def cholesky_py(A):
    
    n = A.shape[0]
    L = np.zeros_like(A)

    for i in range(n):
        for j in range(i + 1):
            if i == j:
                S = 0.                
                for k in range(j):
                    S += L[j, k] * L[j, k]                
                L[j, j] = np.sqrt(A[j, j] - S)

            if i > j:
                S = 0.                
                for k in range(j):
                    S += L[i, k] * L[j, k]                    
                L[i, j] = (A[i, j] - S) / L[j, j]

    return L

We use the following $800 \times 800$ matrix as an example:

In [3]:
n = 800

B = np.random.randn(n, n)
A = B @ B.T  # symmetric and positive definite

In [4]:
np.set_printoptions(precision=3)

cholesky_py(A)

array([[27.564,  0.   ,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 0.056, 28.272,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 1.088,  0.807, 28.743, ...,  0.   ,  0.   ,  0.   ],
       ...,
       [-0.708,  0.039,  0.375, ...,  1.61 ,  0.   ,  0.   ],
       [ 0.52 ,  0.845, -0.341, ...,  1.775,  0.365,  0.   ],
       [-1.056,  0.285, -0.184, ...,  0.119,  1.287,  1.79 ]],
      shape=(800, 800))

**Performance:**

In [5]:
%time cholesky_py(A)

CPU times: user 37.7 s, sys: 6.04 ms, total: 37.7 s
Wall time: 37.9 s


array([[27.564,  0.   ,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 0.056, 28.272,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 1.088,  0.807, 28.743, ...,  0.   ,  0.   ,  0.   ],
       ...,
       [-0.708,  0.039,  0.375, ...,  1.61 ,  0.   ,  0.   ],
       [ 0.52 ,  0.845, -0.341, ...,  1.775,  0.365,  0.   ],
       [-1.056,  0.285, -0.184, ...,  0.119,  1.287,  1.79 ]],
      shape=(800, 800))

### 2. NumPy

We now exploit vectorization in NumPy:

In [6]:
def cholesky_np(A):
    
    n = np.shape(A)[0]
    L = np.zeros_like(A)

    for i in range(n):
        for j in range(i + 1):
            if i == j:
                L[j, j] = np.sqrt(A[j, j] - np.dot(L[j, :j], L[j, :j]))

            if i > j:
                L[i, j] = (A[i, j] - np.dot(L[i, :j], L[j, :j])) / L[j, j]

    return L

Using this for the earlier matrix `A`:

In [7]:
cholesky_np(A)

array([[27.564,  0.   ,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 0.056, 28.272,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 1.088,  0.807, 28.743, ...,  0.   ,  0.   ,  0.   ],
       ...,
       [-0.708,  0.039,  0.375, ...,  1.61 ,  0.   ,  0.   ],
       [ 0.52 ,  0.845, -0.341, ...,  1.775,  0.365,  0.   ],
       [-1.056,  0.285, -0.184, ...,  0.119,  1.287,  1.79 ]],
      shape=(800, 800))

**Performance:**

In [8]:
%time cholesky_np(A)

CPU times: user 657 ms, sys: 2.97 ms, total: 660 ms
Wall time: 659 ms


array([[27.564,  0.   ,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 0.056, 28.272,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 1.088,  0.807, 28.743, ...,  0.   ,  0.   ,  0.   ],
       ...,
       [-0.708,  0.039,  0.375, ...,  1.61 ,  0.   ,  0.   ],
       [ 0.52 ,  0.845, -0.341, ...,  1.775,  0.365,  0.   ],
       [-1.056,  0.285, -0.184, ...,  0.119,  1.287,  1.79 ]],
      shape=(800, 800))

Thus, using NumPy's vectorized operations has significantly improved the performance of the function. 

### 3. Numba

We can improve the performance of the function `cholesky_py` using Numba. 

In [9]:
import numba as nb

In [10]:
cholesky_py_nb = nb.njit(cholesky_py)
cholesky_py_nb(A)

array([[27.564,  0.   ,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 0.056, 28.272,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 1.088,  0.807, 28.743, ...,  0.   ,  0.   ,  0.   ],
       ...,
       [-0.708,  0.039,  0.375, ...,  1.61 ,  0.   ,  0.   ],
       [ 0.52 ,  0.845, -0.341, ...,  1.775,  0.365,  0.   ],
       [-1.056,  0.285, -0.184, ...,  0.119,  1.287,  1.79 ]],
      shape=(800, 800))

**Performance:**

In [11]:
%time cholesky_py_nb(A)

CPU times: user 82.3 ms, sys: 3.84 ms, total: 86.1 ms
Wall time: 89.1 ms


array([[27.564,  0.   ,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 0.056, 28.272,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 1.088,  0.807, 28.743, ...,  0.   ,  0.   ,  0.   ],
       ...,
       [-0.708,  0.039,  0.375, ...,  1.61 ,  0.   ,  0.   ],
       [ 0.52 ,  0.845, -0.341, ...,  1.775,  0.365,  0.   ],
       [-1.056,  0.285, -0.184, ...,  0.119,  1.287,  1.79 ]],
      shape=(800, 800))

This is significantly faster than `cholesky_py`.

Now, we do the same for `cholesky_np`:

In [12]:
cholesky_np_nb = nb.njit(cholesky_np)

cholesky_np_nb(A)

array([[27.564,  0.   ,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 0.056, 28.272,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 1.088,  0.807, 28.743, ...,  0.   ,  0.   ,  0.   ],
       ...,
       [-0.708,  0.039,  0.375, ...,  1.61 ,  0.   ,  0.   ],
       [ 0.52 ,  0.845, -0.341, ...,  1.775,  0.365,  0.   ],
       [-1.056,  0.285, -0.184, ...,  0.119,  1.287,  1.79 ]],
      shape=(800, 800))

**Performance:**

In [13]:
%time cholesky_np_nb(A)

CPU times: user 34.3 ms, sys: 994 μs, total: 35.3 ms
Wall time: 35.3 ms


array([[27.564,  0.   ,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 0.056, 28.272,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 1.088,  0.807, 28.743, ...,  0.   ,  0.   ,  0.   ],
       ...,
       [-0.708,  0.039,  0.375, ...,  1.61 ,  0.   ,  0.   ],
       [ 0.52 ,  0.845, -0.341, ...,  1.775,  0.365,  0.   ],
       [-1.056,  0.285, -0.184, ...,  0.119,  1.287,  1.79 ]],
      shape=(800, 800))

We can see that this is in fact better than the performance we obtained from `cholesky_py_nb`.

### 4. Using `linalg`

The function `np.linalg.cholesky` in NumPy is highly optimized:

In [14]:
np.linalg.cholesky(A)

array([[27.564,  0.   ,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 0.056, 28.272,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 1.088,  0.807, 28.743, ...,  0.   ,  0.   ,  0.   ],
       ...,
       [-0.708,  0.039,  0.375, ...,  1.61 ,  0.   ,  0.   ],
       [ 0.52 ,  0.845, -0.341, ...,  1.775,  0.365,  0.   ],
       [-1.056,  0.285, -0.184, ...,  0.119,  1.287,  1.79 ]],
      shape=(800, 800))

**Performance:**

In [15]:
%time np.linalg.cholesky(A)

CPU times: user 292 ms, sys: 8.86 ms, total: 301 ms
Wall time: 25.2 ms


array([[27.564,  0.   ,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 0.056, 28.272,  0.   , ...,  0.   ,  0.   ,  0.   ],
       [ 1.088,  0.807, 28.743, ...,  0.   ,  0.   ,  0.   ],
       ...,
       [-0.708,  0.039,  0.375, ...,  1.61 ,  0.   ,  0.   ],
       [ 0.52 ,  0.845, -0.341, ...,  1.775,  0.365,  0.   ],
       [-1.056,  0.285, -0.184, ...,  0.119,  1.287,  1.79 ]],
      shape=(800, 800))